# Port Tariff Calculator — exploration

This notebook is a **consumer** of the `tariffs` package. It defines no formulas, no rate values and no logic of its own — everything below just calls the engine and inspects its output. It is optional and not part of the graded path; the tests are (see `tests/`).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from datetime import datetime

from tariffs.engine import calculate
from tariffs.models import VesselCall, Port, VesselType, PeriodBasis
from tariffs.adapter import to_assignment_output

## Reference case: SUDESTADA, Durban

Inputs as reconciled in `SPEC.md` §2 — GT 51,255 (not the vessel sheet's 51,300) and chargeable period 3.396 days (not the sheet's truncated 3.39).

In [3]:
call = VesselCall(
    vessel_name="SUDESTADA",
    port=Port.DURBAN,
    gross_tonnage=51255,
    length_overall_m=229.2,
    vessel_type=VesselType.BULK_CARRIER,
    arrival=datetime(2024, 11, 15, 10, 12),
    departure=datetime(2024, 11, 22, 13, 0),
    chargeable_period_days=3.396,
    chargeable_period_basis=PeriodBasis.DAYS_ALONGSIDE_PROXY,
    number_of_operations=2,
)

result = calculate(call)
result.totals()

{'light_dues': 60062.04,
 'port_dues': 199549.22,
 'towage_dues': 147074.38,
 'vts_dues': 33315.75,
 'pilotage_dues': 47189.94,
 'berthing_services': 19639.5,
 'running_of_vessel_lines': None}

## The full trace

One row per calculation step: the tariff, its section/page reference, the inputs used, the rounding applied, and the subtotal it produced. This is what lets a reviewer confirm the engine is doing what it claims, rather than just calling it.

In [4]:
result.trace_df()

,tariff,section,page,description,inputs,rounding,modifier,modifier_resolution,subtotal
0,Light dues,1.1.1,9,ceil(GT/100) x rate_per_100t (foreign/other ve...,"{'gross_tonnage': 51255.0, 'units': 513, 'rate...",ceil_per_100_t,None,None,60062.04
1,Port dues,4.1.1,21,basic = ceil(GT/100) x basic_rate_per_100t,"{'gross_tonnage': 51255.0, 'units': 513, 'basi...",pro_rata_time,None,None,98870.49
2,Port dues,4.1.1,21,incremental = ceil(GT/100) x incremental_rate_...,"{'chargeable_period_days': 3.396, 'chargeable_...",pro_rata_time,None,None,100678.73
3,Towage dues,3.6,15,"banded_base_plus_increment(GT, durban bands) x...","{'gross_tonnage': 51255.0, 'port': 'durban', '...",ceil_per_100_t,None,None,147074.38
4,VTS dues,2.1.1,11,GT x rate_per_gt (RoundingMode.EXACT — no ceil...,"{'gross_tonnage': 51255.0, 'port': 'durban', '...",exact,None,None,33315.75
5,Pilotage dues,3.3,13,(base_fee + ceil(GT/100) x per_100t) x 2 servi...,"{'gross_tonnage': 51255.0, 'port': 'durban', '...",ceil_per_100_t,None,None,47189.94
6,Berthing services (§3.8),3.8,18,(base_fee + ceil(GT/100) x per_100t) x 2 servi...,"{'gross_tonnage': 51255.0, 'port': 'durban', '...",ceil_per_100_t,None,None,19639.50
7,Running of vessel lines (§3.9),3.9,19,"Parsed, not calculated in v1 (SPEC.md §7.6).",{'mooring_boat_used': None},NaN,None,None,NaN


## Any warnings?

Empty here — the reference case doesn't set `mooring_boat_used`, so no §3.9 warning fires.

In [5]:
result.warnings()

[]

## Assignment-shaped output

The adapter renames domain results onto the assignment's output slots. Note `running_of_vessel_lines` is populated from berthing services (§3.8), with the §3.8/§3.9 mapping note carried in the output itself — and §3.9 is still surfaced separately, never suppressed.

In [6]:
to_assignment_output(result)

{'light_dues': {'amount': 60062.04, 'currency': 'ZAR', 'warnings': []},
 'port_dues': {'amount': 199549.22, 'currency': 'ZAR', 'warnings': []},
 'towage_dues': {'amount': 147074.38, 'currency': 'ZAR', 'warnings': []},
 'vts_dues': {'amount': 33315.75, 'currency': 'ZAR', 'warnings': []},
 'pilotage_dues': {'amount': 47189.94, 'currency': 'ZAR', 'warnings': []},
 'running_of_vessel_lines': {'amount': 19639.5,
  'currency': 'ZAR',
  'warnings': [],
  'note': 'The supplied benchmark value of ZAR 19,639.50 reconciles exactly to Tariff Book §3.8 Berthing Services (Other Ports), not to §3.9 Running of Vessel Lines, which would give ZAR 3,309.12 for two services. The benchmark figure is reported here; both sections are implemented separately in the domain model.'},
 'running_of_vessel_lines_section_3_9_not_calculated': {'amount': None,
  'currency': 'ZAR',
  'warnings': []}}

## A vessel with a mooring boat used

Setting `mooring_boat_used=True` doesn't change any calculated amount in v1 — it adds a warning instead (§3.9 is parsed, not calculated).

In [7]:
call_with_mooring_boat = call.model_copy(update={"mooring_boat_used": True})
result_with_mooring_boat = calculate(call_with_mooring_boat)
result_with_mooring_boat.warnings()

['A §3.9 Running of Vessel Lines charge applies (mooring_boat_used=True) but is not calculated in this version — see SPEC.md §7.6 / README.']